# LoRA hyperparameter sensitivity — analysis

Compares each LoRA sensitivity config (`cosine`, `r32`, `lr1e-4`; seed 42) with the main LoRA run with the same seed
(`lora/seed-42`), on the same Code4ML test instructions and DS-1000 problems:

| Section | Data | Test |
|---|---|---|
| 1 | `metrics.json` of each adapter | validation loss, epochs, minutes, peak VRAM |
| 2 | `summary.json`, `ds1000_results.jsonl` | Pass@1 / Pass@10; exact McNemar vs baseline on binary pass (sample 0 / any of 10), paired bootstrap 95% CI, Holm over the configs |
| 3 | `code4ml_ratings.csv` (from `rating/rate_outputs.ipynb`) | descriptive table; Wilcoxon signed-rank vs baseline per metric, Holm over the configs |
| 4 | main LoRA run, seeds 42/43/44 | seed range as the reference for "no real change" |

Configs whose results are not on the Hub yet are left out.
**Colab:** secret `HF_TOKEN` (read). Locally an existing `hf auth login` is used.

In [7]:
%pip install -q pandas scipy statsmodels huggingface_hub


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os

try:  # Colab secret; elsewhere an existing `hf auth login` is used
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

import json

import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

REPO = "dimsmauls/code4ml-eval-outputs"
HPARAM_REPO = "dimsmauls/qwen2.5-coder-7b-code4ml-hparam"
BASE_ADAPTERS = {"lora": "dimsmauls/qwen2.5-coder-7b-code4ml-lora"}
METHODS = ("lora",)
CONFIGS = ["cosine", "r32", "lr1e-4"]
SEED = 42
ALPHA = 0.05
BOOTSTRAP = 10_000

files = set(HfApi().list_repo_files(REPO, repo_type="dataset"))


def get(path):
    return hf_hub_download(REPO, path, repo_type="dataset")


# (method, config) -> run folder; "baseline" = the main run with the same seed
RUNS = {}
for method in METHODS:
    RUNS[(method, "baseline")] = f"{method}/seed-{SEED}"
    for config in CONFIGS:
        run = f"hp-{method}-{config}/seed-{SEED}"
        if f"{run}/summary.json" in files:
            RUNS[(method, config)] = run
print("evaluated sensitivity runs:", [run for (m, c), run in RUNS.items() if c != "baseline"] or "none yet")

evaluated sensitivity runs: ['hp-lora-cosine/seed-42', 'hp-lora-r32/seed-42', 'hp-lora-lr1e-4/seed-42']


## 1. Training

In [9]:
rows = []
for (method, config) in RUNS:
    if config == "baseline":
        repo, path = BASE_ADAPTERS[method], f"seed-{SEED}/metrics.json"
    else:
        repo, path = HPARAM_REPO, f"{method}-{config}/seed-{SEED}/metrics.json"
    metrics = json.load(open(hf_hub_download(repo, path)))
    rows.append({"method": method, "config": config, "best_eval_loss": metrics["best_eval_loss"],
                 "epochs_run": metrics["epochs_run"], "train_minutes": metrics["train_minutes"],
                 "peak_vram_gb": metrics["peak_vram_gb"], "gpu": metrics["gpu"]})
pd.DataFrame(rows).set_index(["method", "config"]).round(4)

best_eval_loss  epochs_run  train_minutes  peak_vram_gb                    gpu
method config                                                                                  
lora   baseline          1.0839         3.0           12.4         28.72  NVIDIA A100-SXM4-80GB
       cosine            1.0811         3.0           11.9         28.72  NVIDIA A100-PCIE-40GB
       r32               1.0626         3.0           12.2         30.32  NVIDIA A100-PCIE-40GB
       lr1e-4            1.1011         3.0           11.9         30.75  NVIDIA A100-PCIE-40GB

## 2. Functional correctness (DS-1000, 997 problems)
Difference = config − baseline (positive = the config is better).

In [10]:
rng = np.random.default_rng(42)


def bootstrap_ci(diffs):
    diffs = np.asarray(diffs, dtype=float)
    means = diffs[rng.integers(0, len(diffs), (BOOTSTRAP, len(diffs)))].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])


def binary(run):
    res = pd.read_json(get(f"{run}/ds1000_results.jsonl"), lines=True)
    return pd.DataFrame({"pass1": res[res["sample"] == 0].set_index("problem_id")["passed"].astype(int),
                         "pass10": res.groupby("problem_id")["passed"].any().astype(int)})


summaries = {key: json.load(open(get(f"{run}/summary.json"))) for key, run in RUNS.items()}
binaries = {key: binary(run) for key, run in RUNS.items()}

families = []
for method in METHODS:
    base = binaries[(method, "baseline")]
    family = []
    for config in CONFIGS:
        if (method, config) not in RUNS:
            continue
        row = {"method": method, "config": config,
               "pass@1": summaries[(method, config)]["ds1000"]["pass@1"],
               "pass@1 baseline": summaries[(method, "baseline")]["ds1000"]["pass@1"],
               "pass@10": summaries[(method, config)]["ds1000"]["pass@10"],
               "pass@10 baseline": summaries[(method, "baseline")]["ds1000"]["pass@10"]}
        for k in ("pass1", "pass10"):
            joined = binaries[(method, config)][[k]].join(base[[k]], rsuffix="_base", how="inner")
            a, b = joined[k], joined[f"{k}_base"]
            only_a, only_b = int(((a == 1) & (b == 0)).sum()), int(((a == 0) & (b == 1)).sum())
            both = int(((a == 1) & (b == 1)).sum())
            p = mcnemar([[both, only_a], [only_b, len(joined) - both - only_a - only_b]], exact=True).pvalue
            low, high = bootstrap_ci(a - b)
            row.update({f"{k} diff": (a - b).mean(), f"{k} ci95_low": low, f"{k} ci95_high": high, f"{k} p": p})
        family.append(row)
    if family:
        family = pd.DataFrame(family)
        for k in ("pass1", "pass10"):
            family[f"{k} p_holm"] = multipletests(family[f"{k} p"], method="holm")[1]
        families.append(family)
functional = pd.concat(families).set_index(["method", "config"]) if families else pd.DataFrame()
functional.round(4)

pass@1  pass@1 baseline  pass@10  pass@10 baseline  pass1 diff  pass1 ci95_low  pass1 ci95_high  pass1 p  pass10 diff  pass10 ci95_low  pass10 ci95_high  pass10 p  pass1 p_holm  pass10 p_holm
method config                                                                                                                                                                                                 
lora   cosine  0.3086           0.3168   0.5968            0.6179     -0.0030         -0.0311           0.0241   0.8900      -0.0211          -0.0441            0.0020    0.0941        0.8900         0.2582
       r32     0.3247           0.3168   0.6108            0.6179      0.0150         -0.0140           0.0431   0.3329      -0.0070          -0.0301            0.0171    0.6208        0.6657         0.6208
       lr1e-4  0.3223           0.3168   0.6399            0.6179      0.0241         -0.0050           0.0532   0.1208       0.0221          -0.0020            0.0461    0.0861        0.3624         0.2582

## 3. Structural quality (Code4ML test, 304 instructions)
Needs `code4ml_ratings.csv` of each run (run `rating/rate_outputs.ipynb` first). Difference = config − baseline;
for ratings and counts a **negative** difference means the config is better.

In [11]:
GRADE = {g: i for i, g in enumerate("ABCDE", 1)}
METRICS = ["maintainability", "code_smells", "complexity", "cognitive_complexity", "bugs", "code_smells_per_100_ncloc"]


def ratings(run):
    df = pd.read_csv(get(f"{run}/code4ml_ratings.csv"), dtype={"id": str}).set_index("id")
    valid = df["ast_valid"].astype(str) == "True"
    ncloc = pd.to_numeric(df["ncloc"], errors="coerce")
    analysed = valid & (ncloc > 0)                     # SonarQube N/A without ncloc
    out = pd.DataFrame(index=df.index)
    out["ast_valid"] = valid
    out["ncloc"] = ncloc.where(analysed)
    out["maintainability"] = df["maintainability_rating"].map(GRADE).where(analysed)
    for metric in ("code_smells", "complexity", "cognitive_complexity", "bugs"):
        out[metric] = pd.to_numeric(df[metric], errors="coerce").where(analysed)
    out["code_smells_per_100_ncloc"] = 100 * out["code_smells"] / out["ncloc"]
    return out


rated = {key: ratings(run) for key, run in RUNS.items() if f"{run}/code4ml_ratings.csv" in files}
print("not rated yet:", [run for key, run in RUNS.items() if key not in rated] or "none")

display(pd.DataFrame({f"{method} | {config}": {
    "ast_valid_rate": df["ast_valid"].mean(), "median_ncloc": df["ncloc"].median(),
    "maintainability_A_%": 100 * (df["maintainability"] == 1).sum() / df["maintainability"].notna().sum(),
    **{f"mean_{m}": df[m].mean() for m in METRICS}} for (method, config), df in rated.items()}).T.round(3))

families = []
for method in METHODS:
    if (method, "baseline") not in rated:
        continue
    base = rated[(method, "baseline")]
    for metric in METRICS:
        family = []
        for config in CONFIGS:
            if (method, config) not in rated:
                continue
            diff = (rated[(method, config)][metric] - base[metric]).dropna()
            nonzero = diff[diff != 0]
            p = stats.wilcoxon(nonzero).pvalue if len(nonzero) else 1.0
            family.append({"method": method, "metric": metric, "config": config, "n": len(diff),
                           "mean_diff": diff.mean(), "p": p})
        if family:
            family = pd.DataFrame(family)
            family["p_holm"] = multipletests(family["p"], method="holm")[1]
            families.append(family)
if families:
    structural = pd.concat(families)
    structural["significant"] = structural["p_holm"] < ALPHA
    display(structural.set_index(["method", "metric", "config"]).round(4))
else:
    print("no sensitivity run is rated yet")

not rated yet: ['hp-lora-cosine/seed-42', 'hp-lora-r32/seed-42', 'hp-lora-lr1e-4/seed-42']


,ast_valid_rate,median_ncloc,maintainability_A_%,mean_maintainability,mean_code_smells,mean_complexity,mean_cognitive_complexity,mean_bugs,mean_code_smells_per_100_ncloc
lora | baseline,0.99,2.0,96.013,1.04,0.226,0.003,0.0,0.0,4.651


no sensitivity run is rated yet


## 4. Reference: variation between seeds of the main runs
A config difference inside this range is not distinguishable from changing the seed.

In [12]:
rows = []
for method in METHODS:
    seeds = [json.load(open(get(f"{method}/seed-{s}/summary.json")))["ds1000"] for s in (42, 43, 44)]
    rows.append({"method": method,
                 "pass@1 min": min(s["pass@1"] for s in seeds), "pass@1 max": max(s["pass@1"] for s in seeds),
                 "pass@10 min": min(s["pass@10"] for s in seeds), "pass@10 max": max(s["pass@10"] for s in seeds)})
pd.DataFrame(rows).set_index("method").round(4)

,pass@1 min,pass@1 max,pass@10 min,pass@10 max
method,,,,
lora,0.3086,0.3175,0.6058,0.6179
